# Basic MjSpec Examples
- Use mjSpec to generate MuJoCo model

In [ ]:

import os
import sys
import numpy as np
import time
import mujoco
sys.path.append(os.path.abspath('../'))
# from pp_base_mujoco.VIEWER import MUJOCOGLVIEWER
from pp_base_mujoco.VIEWER import *

import xml.etree.ElementTree as ET
from lxml import etree

In [ ]:
def print_xml(xml_input,color=True):
    if isinstance(xml_input, ET.Element):
        rough_string = ET.tostring(xml_input, encoding='unicode')
    else:
        rough_string = xml_input

    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.fromstring(rough_string, parser=parser)
    pretty_xml = etree.tostring(tree, pretty_print=True, encoding='unicode')
    print(pretty_xml)

#### 1. Declare Spec: empty spec / from string

In [ ]:
# simple spec
spec = mujoco.MjSpec()
print(spec) # mj spec object
print_xml(spec.to_xml())

In [ ]:
path = '../asset/floor_white_gray.xml'
spec = mujoco.MjSpec.from_file(path)

#### 2. Add
- add body: adding empty body
    - add geom: adding geom

In [ ]:
# add specific body to worldbody
body = spec.worldbody.add_body(
    name="base_link",
    pos = np.zeros((3)),
    euler = [0, 0.8, 0] # auto-fixed to quat
)
print_xml(spec.to_xml())

print(body) # returns spec body object
print(body == spec.body("base_link"))

In [ ]:
# add geom to body
geom = body.add_geom(
    name="base_link_geom",
    pos = [0.0, 0.0, 0.1],
    type=mujoco.mjtGeom.mjGEOM_BOX,
    size = [0.1, 0.1, 0.1],
    rgba=[0,1,0,1]
)
spec.body("base_link").add_site(
    name="base_link_site"
)

print(geom)
print(body)
print_xml(spec.to_xml())

In [ ]:
# add hierarchical body -> under body 1
body2 = body.add_body(
    name="link1",
    pos = [0.0,0.0,0.3],
)

body2.add_geom(
    name="link1_geom",
    pos = [0.0,0.0,0.1],
    type=mujoco.mjtGeom.mjGEOM_SPHERE,
    size = [0.1, 0.0, 0.0],
    rgba=[1,0,0,1]
)

print_xml(spec.to_xml())

#### 3. Frame & Attach

In [ ]:
frame1 = spec.worldbody.add_frame(pos=[0,3,3], quat=[0, 0, 0, 1])

arena_xml = """
<mujoco>
<worldbody>
    <body name="box" pos="0 0 0">
        <geom type="box" size="1 1 1"/>
    </body>
</worldbody>
</mujoco>
"""

additional_spec = mujoco.MjSpec.from_string(arena_xml)
body3 = additional_spec.body('box')

body4 = frame1.attach_body(body3, 'attached-', '-1') # body object, prefix, postfix

print_xml(spec.to_xml())

In [ ]:
model = spec.compile()
data = mujoco.MjData(model)

In [ ]:
""" MAIN LOOP """

# create python viewer object
viewer = MUJOCOGLVIEWER(model, data)

# data reset
mujoco.mj_resetData(model, data)

while viewer.is_alive():
        mujoco.mj_step(model, data)
        viewer.render()

# close
viewer.close()